In [218]:
from ultralytics import YOLO

model = YOLO("C:/Users/LENOVO/Retail-Shelf-Monitoring/runs/baseline/weights/best.pt")
print(model.names)



{0: 'object'}


In [219]:
source = "C:/Users/LENOVO/Retail-Shelf-Monitoring/data/samples/test_1000.jpg"  # or a folder path
results = model(
    source,
    max_det=1000,
    conf=0.25,
    save=True,
    project="C:/Users/LENOVO/Retail-Shelf-Monitoring/predictions",
    name="test_0",
)


image 1/1 C:\Users\LENOVO\Retail-Shelf-Monitoring\data\samples\test_1000.jpg: 640x480 140 objects, 52.1ms
Speed: 3.7ms preprocess, 52.1ms inference, 3.9ms postprocess per image at shape (1, 3, 640, 480)
Results saved to C:\Users\LENOVO\Retail-Shelf-Monitoring\predictions\test_0-7


In [220]:
results[0].orig_shape

(3264, 2448)

In [221]:
bounding_box = results[0].boxes.xyxy
bounding_box


tensor([[9.4401e+02, 2.5555e+03, 1.0685e+03, 2.8871e+03],
        [9.8576e+02, 6.5873e+02, 1.1339e+03, 8.7271e+02],
        [1.1468e+03, 6.6778e+02, 1.2934e+03, 8.8963e+02],
        [1.0768e+03, 2.5570e+03, 1.1951e+03, 2.8696e+03],
        [1.3038e+03, 6.7810e+02, 1.4549e+03, 8.9842e+02],
        [9.1390e+01, 1.2511e+03, 2.2340e+02, 1.6397e+03],
        [0.0000e+00, 1.2508e+03, 7.9560e+01, 1.6445e+03],
        [2.3975e+02, 2.1846e+03, 3.4905e+02, 2.5033e+03],
        [6.5745e+02, 6.2767e+02, 8.1510e+02, 8.4791e+02],
        [1.2038e+03, 2.5307e+03, 1.3223e+03, 2.8479e+03],
        [1.4515e+03, 2.5058e+03, 1.5637e+03, 2.8140e+03],
        [8.1971e+02, 1.2695e+03, 9.4159e+02, 1.4742e+03],
        [7.6917e+01, 3.2966e+02, 2.0348e+02, 7.3340e+02],
        [6.4323e+02, 1.2692e+03, 8.1072e+02, 1.4937e+03],
        [1.2266e+03, 2.1022e+03, 1.3225e+03, 2.4013e+03],
        [8.3026e+02, 6.3849e+02, 9.7900e+02, 8.5767e+02],
        [1.5696e+03, 2.4995e+03, 1.6855e+03, 2.7938e+03],
        [3.560

In [222]:
# bounding_box_sorted = sorted(
#     bounding_box,
#     key=lambda box: box[0]
# )


In [223]:
# y2 = [box[3] for box in bounding_box_sorted]


In [224]:
# y2

In [225]:

# groups = [[]]

# for j in range(len(y2)):

#     if j > 0 and y2[j] - y2[j-1] > 200:
#         groups.append([])

#     groups[-1].append(bounding_box_sorted[j])

# groups

In [226]:
# SEUIL = 0# pixels

# for numero, etagere in enumerate(groups, start=1):
#     etagere_triee = sorted(etagere, key=lambda box: box[0])   # tri par x1
#     for k in range(1, len(etagere_triee)):
#         gauche = etagere_triee[k-1]
#         droite = etagere_triee[k]
#         ecart = droite[0] - gauche[2]                         # x1(droite) - x2(gauche)
#         if ecart > SEUIL:
#             print(f"Produit absent dans l'étagère {numero} ")

In [227]:
# my_list = []
# bounding_box_sorted = sorted(bounding_box, key=lambda box: box[3])
# for i in range(len(bounding_box_sorted)):
#     my_list.append(float(bounding_box_sorted[i][3]))
# my_list

In [228]:
bounding_box_sorted = sorted(bounding_box, key=lambda box: box[3])



In [229]:
import numpy as np
from sklearn.cluster import DBSCAN
import cv2

# Height de l'image
image = cv2.imread(source)

H = image.shape[0]

# Extraire et normaliser y2
y2_normalized = np.array([
    box[3] / H for box in bounding_box_sorted
]).reshape(-1, 1)

# DBSCAN
dbscan = DBSCAN(
    eps=0.04,
    min_samples=2
)

labels = dbscan.fit_predict(y2_normalized)

print("Clusters:", set(labels))

Clusters: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)}


In [230]:
clusters = set(labels)

bounding_box_sorted = np.array(bounding_box_sorted)

cluster_0 = bounding_box_sorted[labels == 0]

# for i in range(bounding_box_sorted) :

cluster_0[0][0]


np.float32(0.0)

In [231]:
sorted_shelves = {}

for i in set(labels):
    if i == -1:
        continue

    cluster = bounding_box_sorted[labels == i]

    sorted_shelves[i] = sorted(cluster, key=lambda box: box[0])

sorted_shelves[0][0]

array([          0,      321.19,      69.784,      727.92], dtype=float32)

In [232]:
SEUIL = 100

for shelf_id, products in sorted_shelves.items():

    for j in range(len(products) - 1):

        current_product = products[j]
        next_product = products[j + 1]

        gap = next_product[0] - current_product[2]

        if gap > SEUIL:
            print(f"Gap détecté dans l'étagère {shelf_id} : {gap:.2f} pixels")

Gap détecté dans l'étagère 3 : 103.77 pixels
Gap détecté dans l'étagère 4 : 236.98 pixels
Gap détecté dans l'étagère 5 : 111.64 pixels
